# Conclusão geral

## Objetivo e hipótese

Este estudo investigou se é possível identificar pessoas com depressão usando informações do NHIS 2025 sem fornecer aos modelos os próprios sintomas de depressão ou de ansiedade. A pergunta é relevante porque, em um cenário de rastreio, pode ser útil reconhecer sinais associados à depressão mesmo quando não há respostas diretas a questionários de saúde mental.

Foram comparados dois cenários: **A**, exploratório, que mantém as variáveis de ansiedade; e **B**, principal para a hipótese, que remove os sintomas e demais variáveis de depressão (PHQ) e ansiedade (GAD-7). O cenário A serve como referência de desempenho quando há informação de ansiedade, que costuma estar associada à depressão. A hipótese central é avaliada pelo cenário B.

## Como a comparação foi conduzida

- Os quatro algoritmos — Random Forest, KNN, SVM e XGBoost — foram avaliados com os mesmos 22.084 registros completos e o mesmo split estratificado de treino e teste (80/20, `random_state=42`), para tornar os resultados comparáveis.
- O desfecho foi tratado como classificação binária: registros sem depressão versus registros com depressão. Respostas inválidas ou indefinidas no alvo foram excluídas.
- O peso amostral não foi usado para treinar os modelos; foi usado nas métricas ponderadas. As matrizes de confusão e contagens de falsos positivos e falsos negativos são contagens de registros, sem multiplicação pelo peso.
- A escolha de hiperparâmetros e dos limiares de recall foi feita sem usar o conjunto de teste. Os limiares para recall-alvo foram escolhidos no treino, com validação cruzada; portanto, o recall efetivamente obtido no teste pode não ser exatamente o alvo.

## O que aprendemos sobre a hipótese

A retirada das variáveis de ansiedade piorou o desempenho, como esperado, mas **não eliminou a capacidade preditiva**. No cenário B, todos os modelos ficaram bem acima da referência de acaso (ROC-AUC de 0,50). Isso sustenta a hipótese de que informações não diretamente relacionadas a sintomas de saúde mental carregam sinal útil para prever depressão.

| Modelo no cenário B | ROC-AUC | ROC-AUC sem cognição e sem solidão |
|---|---:|---:|
| SVM linear ajustado | 0,856 | 0,813 |
| XGBoost ajustado | 0,856 | 0,811 |
| KNN com colunas ponderadas | 0,843 | 0,796 |
| Random Forest | 0,841 | 0,804 |
| KNN ajustado, sem ponderação de colunas | 0,816 | 0,778 |

No cenário A, SVM, XGBoost e Random Forest alcançaram ROC-AUC de aproximadamente 0,915; o KNN original ficou em 0,833. A queda ao retirar ansiedade foi, portanto, esperada e relevante, mas os modelos ainda mantiveram discriminação substancial no cenário B.

Os sinais observados incluem idade, autoavaliação de saúde, dor crônica, limitações funcionais, suporte social e condições socioeconômicas. Cognição e solidão também aparecem como preditores importantes. Ao removê-las, o desempenho diminui, mas continua acima do acaso, o que mostra que o resultado não depende exclusivamente dessas variáveis.

**Interpretação cuidadosa:** o cenário B demonstra que variáveis sem sintomas diretos de depressão/ansiedade podem ajudar a prever depressão. Ele **não prova que doenças físicas, isoladamente, sejam suficientes**, porque os modelos também usam idade, cognição, funcionamento, relações sociais e outras características. Para testar especificamente a hipótese de doenças físicas, seria necessário comparar um cenário restrito a variáveis físicas com cenários sociais e funcionais separados.

## Comparação dos algoritmos e aprendizado sobre recall

O SVM linear e o XGBoost tiveram o maior ROC-AUC no cenário B (0,856), mas a diferença para Random Forest (0,841) e KNN ponderado (0,843) é pequena em um único split. O empate entre um modelo linear e um modelo mais flexível sugere que, neste conjunto de variáveis, maior complexidade não trouxe uma vantagem clara. Isso não demonstra que o sinal seja exclusivamente linear; a confirmação exigiria validações repetidas e externas.

Em saúde, deixar de identificar um caso pode ser mais custoso do que encaminhar alguém sem depressão para uma avaliação adicional. Por isso, também foram testados pontos de operação com recall elevado. A tabela mostra o resultado no teste ao selecionar no treino um limiar para recall-alvo de 90%:

| Modelo | Recall obtido | Precisão ponderada | Falsos positivos (contagem) | Amostra sinalizada |
|---|---:|---:|---:|---:|
| SVM linear | 86,9% | 38,5% | 1.341 | 48,9% |
| XGBoost | 88,4% | 38,8% | 1.393 | 50,5% |
| Random Forest | 88,3% | 38,0% | 1.445 | 51,7% |
| KNN com colunas ponderadas | 88,0% | 37,0% | 1.530 | 53,7% |
| KNN ajustado | 87,2% | 34,0% | 1.679 | 56,8% |

O recall maior tem um custo: aproximadamente metade da amostra é sinalizada e há muitos falsos positivos. A precisão fica entre 34% e 39%, embora seja superior à prevalência do desfecho (cerca de 21%). O limiar não é uma escolha puramente estatística: deve refletir a capacidade de acompanhamento, o custo de falsos positivos e o risco de casos perdidos. Os resultados também mostram que o recall medido no teste ficou alguns pontos abaixo do alvo de 90%, pois o limiar foi selecionado sem consultar o teste.

As tentativas de aumentar o peso da classe positiva no Random Forest, SVM e XGBoost elevaram o recall no limiar padrão, mas não melhoraram de forma clara a curva de precisão-recall: elas principalmente mudaram o ponto de operação, aumentando falsos positivos. A mudança mais consistente na qualidade do ranking ocorreu no KNN: ponderar as colunas pelos coeficientes de um SVM L1 elevou o ROC-AUC de 0,816 para 0,843 e o PR-AUC de 0,600 para 0,646; o recall no limiar padrão passou de 45,6% para 66,8%. Ainda assim, no recall-alvo de 90%, essa variante precisou de mais falsos positivos que SVM, XGBoost e Random Forest.

## Limitações e próximos passos

- A avaliação final usa um único split de treino e teste. Pequenas diferenças entre modelos podem resultar da variação da amostra; repetir a validação ou usar validação cruzada externa daria estimativas mais estáveis.
- A análise inclui somente registros completos (22.084 de 24.215), o que pode limitar a representatividade se a ausência de respostas não for aleatória.
- Cognição e solidão podem se sobrepor a manifestações ou consequências da depressão. Sua exclusão é uma análise de sensibilidade, não uma prova de que todo o sinal restante venha de doenças físicas.
- As métricas agregadas não demonstram desempenho equivalente entre grupos demográficos. Uma aplicação futura deve avaliar calibração, taxas de erro por subgrupo e validade em outra amostra e período.
- Um sistema de rastreio exigiria validação externa, revisão clínica e definição explícita do limiar em função dos recursos e consequências de cada tipo de erro. Os modelos deste estudo não devem ser interpretados como diagnóstico.
- Os resultados indicam **associação preditiva**, não causalidade: não demonstram que dor, doença física ou qualquer outra variável cause depressão.

## Conclusão

No NHIS 2025 analisado, informações sem sintomas diretos de depressão e ansiedade conservaram capacidade preditiva relevante, inclusive após retirar cognição e solidão. Isso apoia a hipótese de que características físicas, funcionais e sociais podem contribuir para identificar pessoas com depressão e justifica investigar mais profundamente o papel de dor crônica, limitações funcionais e autoavaliação de saúde.

Nenhum algoritmo foi claramente superior em todos os critérios. SVM e XGBoost tiveram o melhor ROC-AUC; o Random Forest ficou próximo; e o KNN melhorou de forma mais evidente após ponderação das variáveis. Para priorizar a identificação de casos, o ajuste do limiar foi mais importante que trocar de algoritmo, mas o ganho de recall veio acompanhado de muitos falsos positivos. Assim, os achados são promissores como evidência de viabilidade preditiva e ponto de partida para um estudo mais específico — não como prova de que dados exclusivamente físicos bastam, nem como ferramenta clínica pronta para uso.
